# Task 56: Neural network from scratch

Reference: https://www.analyticsvidhya.com/blog/2020/07/neural-networks-from-scratch-in-python-and-r/

A from-scratch XOR demonstration; the reported accuracy is on the four training truth-table inputs.

In [1]:
# Task 56: Neural network from scratch
# Reference: https://www.analyticsvidhya.com/blog/2020/07/neural-networks-from-scratch-in-python-and-r/

import numpy as np

# XOR needs a nonlinear hidden layer; a single linear boundary is insufficient.
x = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = np.array([[0.], [1.], [1.], [0.]])
rng = np.random.default_rng(13)
w1 = rng.normal(0, 1, (2, 4))
b1 = np.zeros((1, 4))
w2 = rng.normal(0, 1, (4, 1))
b2 = np.zeros((1, 1))
learning_rate = 0.5


def sigmoid(values):
    return 1 / (1 + np.exp(-np.clip(values, -50, 50)))


print('Architecture: 2 inputs -> 4 sigmoid hidden units -> 1 sigmoid output')
for epoch in range(1, 6001):
    # Forward propagation.
    hidden = sigmoid(x @ w1 + b1)
    probabilities = sigmoid(hidden @ w2 + b2)
    safe = np.clip(probabilities, 1e-8, 1 - 1e-8)
    loss = -np.mean(y * np.log(safe) + (1 - y) * np.log(1 - safe))

    # Backpropagation: sigmoid + binary cross-entropy simplifies d(logits).
    d_output = (probabilities - y) / len(x)
    dw2 = hidden.T @ d_output
    db2 = d_output.sum(axis=0, keepdims=True)
    d_hidden = (d_output @ w2.T) * hidden * (1 - hidden)
    dw1 = x.T @ d_hidden
    db1 = d_hidden.sum(axis=0, keepdims=True)
    w1 -= learning_rate * dw1
    b1 -= learning_rate * db1
    w2 -= learning_rate * dw2
    b2 -= learning_rate * db2
    if epoch in [1, 1000, 3000, 6000]:
        print(f'Epoch {epoch}: binary cross-entropy={loss:.6f}')

probabilities = sigmoid(sigmoid(x @ w1 + b1) @ w2 + b2)
predictions = (probabilities >= 0.5).astype(int)
print('Training truth table:')
for row, target, probability, prediction in zip(x, y.ravel(), probabilities.ravel(), predictions.ravel()):
    print(f'{row.astype(int).tolist()} target={int(target)}, probability={probability:.4f}, prediction={prediction}')
print('Truth-table accuracy:', float(np.mean(predictions == y)))


Architecture: 2 inputs -> 4 sigmoid hidden units -> 1 sigmoid output
Epoch 1: binary cross-entropy=0.697414
Epoch 1000: binary cross-entropy=0.031627
Epoch 3000: binary cross-entropy=0.006923
Epoch 6000: binary cross-entropy=0.003127
Training truth table:
[0, 0] target=0, probability=0.0006, prediction=0
[0, 1] target=1, probability=0.9970, prediction=1
[1, 0] target=1, probability=0.9966, prediction=1
[1, 1] target=0, probability=0.0055, prediction=0
Truth-table accuracy: 1.0
